In [1]:
from docx import Document
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [2]:
import os

print(os.listdir("data/raw"))

['business _opportunities _india.docx', 'drone _use _cases _and_ success_ stories.docx', 'drone_ Regulations _in_ India.docx', 'technical _comparison _drone_specifications.docx']


In [3]:
def load_docx(folder_path):
    documents = []
    
    for file in os.listdir(folder_path):
        if file.endswith(".docx"):
            doc = Document(os.path.join(folder_path, file))
            text = "\n".join([para.text for para in doc.paragraphs])
            documents.append(text)
    
    return documents

docs = load_docx("data/raw")

print("Documents loaded:", len(docs))

Documents loaded: 4


In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.create_documents(docs)

print("Total chunks created:", len(chunks))

Total chunks created: 15


In [5]:
#embedding.py

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [6]:
#vectorbases
vectorstore = FAISS.from_documents(chunks, embeddings)

print("Vector database created successfully!")

Vector database created successfully!


In [7]:
query = "Explain drone regulations in India"

results = vectorstore.similarity_search(query, k=1)

for doc in results:
    
    print(doc.page_content)

Drone Regulations in India
Introduction
Drone operations in India are governed by the Drone Rules 2021 issued by the Ministry of Civil Aviation and implemented by the Directorate General of Civil Aviation (DGCA). These rules aim to balance innovation, economic growth, and national security. The regulatory framework simplifies drone approvals while ensuring safety, accountability, and airspace control.
Classification of Drones
Drones are categorized based on Maximum Take-Off Weight (MTOW):


In [8]:
#generation
import requests
import json

API_KEY = "Create_your_API_key"      #mandatory to create api key from gemini to connect with llm

def generate_with_gemini(prompt):
    
    url = f"https://generativelanguage.googleapis.com/v1/models/gemini-2.5-flash:generateContent?key={API_KEY}"
    
    headers = {
        "Content-Type": "application/json"
    }
    
    data = {
        "contents": [
            {
                "parts": [
                    {"text": prompt}
                ]
            }
        ]
    }
    
    response = requests.post(url, headers=headers, data=json.dumps(data))
    
    result = response.json()
    
    return result

In [10]:
def ask_rag(query):
    
    # Step 1: Retrieve from FAISS
    results = vectorstore.similarity_search(query, k=3)
    
    context = ""
    for doc in results:
        context += doc.page_content + "\n\n"
    
    # Step 2: Create prompt
    prompt = f"""
    You are an AI assistant for India's Drone Intelligence System.
    
    Use the context below to answer clearly and professionally.
    
    Context:
    {context}
    
    Question:
    {query}
    """
    
    # Step 3: Send to Gemini (Generation Component)
    final_answer = generate_with_gemini(prompt)
    
    return final_answer

In [12]:
answer = ask_rag("Explain type of drons")
print(answer)

{'candidates': [{'content': {'parts': [{'text': "Based on the classifications for India's Drone Intelligence System, drones are categorized by weight as follows:\n\n*   **Nano:** Less than 250 grams\n*   **Micro:** 250 grams to 2 kilograms\n*   **Small:** 2 to 25 kilograms\n*   **Medium:** 25 to 150 kilograms\n*   **Large:** Above 150 kilograms\n\nEach of these categories has distinct requirements regarding registration, licensing, and operational permissions."}], 'role': 'model'}, 'finishReason': 'STOP', 'index': 0}], 'usageMetadata': {'promptTokenCount': 259, 'candidatesTokenCount': 105, 'totalTokenCount': 448, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 259}], 'thoughtsTokenCount': 84}, 'modelVersion': 'gemini-2.5-flash', 'responseId': 'PzulaaSnBK-74-EP2r2ToAI'}
